# FinReasoning AI — Full Training Pipeline

**Base model:** Qwen2.5-14B-Instruct  
**PEFT method:** QLoRA (4-bit NF4, double quantization)  
**Training:** SFT (primary) + optional DPO  
**Hardware requirement:** A100 80GB GPU (Colab Pro+ with A100 runtime)

---

## Pipeline Overview

| Step | What it does |
|------|--------------|
| 0 | Setup: install dependencies, clone repo, mount Drive |
| 1 | Load Qwen2.5-14B in 4-bit NF4 + attach QLoRA adapters |
| 2 | Generate synthetic training data + preprocess into ChatML format |
| 3 | SFT training with TRL SFTTrainer |
| 4 | Evaluate: Exact Match, F1, parsability, grounding rate |
| 5 | Inference: direct answer, CoT, self-consistency, tool use |
| 6 | Extensions: RAG retrieval, ReAct agent (optional) |

**Important:** Change the Colab runtime to **A100 GPU** before running.  
Runtime > Change runtime type > A100 GPU

---
## Step 0 — Environment Setup

In [ ]:
# Verify GPU availability and VRAM
import subprocess, sys

result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
                         '--format=csv,noheader'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError("No GPU detected. Switch runtime to A100 GPU before continuing.")

gpu_info = result.stdout.strip()
print("GPU detected:", gpu_info)

# Parse free VRAM
parts = gpu_info.split(',')
free_mb = int(parts[2].strip().replace(' MiB', ''))
if free_mb < 50000:
    print(f"WARNING: Only {free_mb} MiB free VRAM. Recommend A100 80GB (81920 MiB).")
    print("Training will still run but may require reducing batch_size to 2.")
else:
    print(f"VRAM OK: {free_mb} MiB free.")

In [ ]:
# Mount Google Drive to persist checkpoints and data across sessions
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/FinReasoningAI'
import os
os.makedirs(DRIVE_BASE, exist_ok=True)
print(f"Drive mounted. Outputs will be saved to: {DRIVE_BASE}")

In [ ]:
"""
Dependency installer - fast, conflict-free, Colab-aware.

Strategy:
  - DO NOT reinstall PyTorch. Colab ships a CUDA-optimized build
    (currently torch 2.10.0+cu128). Reinstalling a different version
    causes torchvision/torchaudio conflicts and wastes 5+ minutes.
  - Only install what Colab does not already provide, or where the
    installed version is below the required minimum.
  - bitsandbytes >= 0.44.0 is mandatory: earlier builds import from
    triton.ops which was removed in Triton 3.x (Colab default).

After first install: Runtime > Restart session, then continue from
the next cell. The install cell is idempotent -- re-running it is safe
and fast (already-satisfied requirements are skipped automatically).
"""

import importlib.metadata, subprocess, sys

# ---------------------------------------------------------------------------
# Minimum required versions  (torch is intentionally absent -- use Colab's)
# ---------------------------------------------------------------------------
REQUIRED = {
    "transformers":   "4.41.0",   # 4.41+ needed for Qwen2.5 and sentence-transformers
    "datasets":       "2.19.0",
    "accelerate":     "0.30.0",
    "peft":           "0.10.0",
    "trl":            "0.8.6",
    "bitsandbytes":   "0.44.0",   # CRITICAL: drops broken triton.ops import
    "evaluate":       "0.4.1",
    "rouge-score":    "0.1.2",
    "scikit-learn":   "1.4.0",
    "pandas":         "2.2.0",
    "pydantic":       "2.7.0",
    "jsonlines":      "4.0.0",
}

def _parse(v):
    """Parse a version string into a comparable tuple of ints."""
    return tuple(int(x) for x in v.split(".")[:3] if x.isdigit())

def _installed(pkg):
    try:
        return importlib.metadata.version(pkg)
    except importlib.metadata.PackageNotFoundError:
        return None

def _pip_install(specs):
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + specs
    )

# ---------------------------------------------------------------------------
# Check which packages need installing / upgrading
# ---------------------------------------------------------------------------
to_install = []
already_ok  = []

for pkg, min_ver in REQUIRED.items():
    current = _installed(pkg)
    if current is None or _parse(current) < _parse(min_ver):
        to_install.append(f"{pkg}>={min_ver}")
        status = f"MISSING" if current is None else f"upgrade {current} -> >={min_ver}"
    else:
        already_ok.append(pkg)
        status = f"ok ({current})"
    print(f"  {pkg:<20} {status}")

# ---------------------------------------------------------------------------
# Install only what is needed
# ---------------------------------------------------------------------------
if to_install:
    print(f"\nInstalling {len(to_install)} package(s)...")
    _pip_install(to_install)
    print("Install complete.")
else:
    print("\nAll packages already satisfy minimum versions. Nothing to install.")

# ---------------------------------------------------------------------------
# Hard check: bitsandbytes must be >= 0.44.0 or the model cannot load
# ---------------------------------------------------------------------------
bnb_ver = importlib.metadata.version("bitsandbytes")
_maj, _min, *_ = bnb_ver.split(".")
if int(_maj) == 0 and int(_min) < 44:
    raise RuntimeError(
        f"bitsandbytes {bnb_ver} is installed but >= 0.44.0 is required.\n"
        "This version imports from triton.ops which no longer exists in Triton 3.x.\n"
        "Fix: pip install -U 'bitsandbytes>=0.44.0'  then Runtime > Restart session."
    )

# ---------------------------------------------------------------------------
# Verify PyTorch (must already be present -- we do not install it)
# ---------------------------------------------------------------------------
try:
    import torch
    print(f"\nPyTorch version : {torch.__version__}")
    print(f"CUDA available  : {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU             : {torch.cuda.get_device_name(0)}")
        print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB")
except ImportError:
    raise RuntimeError(
        "PyTorch is not installed. This notebook requires a Colab GPU runtime.\n"
        "Go to Runtime > Change runtime type and select a GPU."
    )

# ---------------------------------------------------------------------------
# flash-attn: optional, build-heavy, skip if it fails
# ---------------------------------------------------------------------------
if _installed("flash-attn") is None:
    print("\nAttempting to install flash-attn (optional, ~2 min build time)...")
    try:
        _pip_install(["flash-attn", "--no-build-isolation"])
        print("flash-attn installed.")
    except Exception as e:
        print(f"flash-attn build failed ({e.__class__.__name__}). Continuing with eager attention.")
else:
    print(f"flash-attn already installed ({_installed('flash-attn')}).")

print("\nSetup complete.")
print("If this was a fresh install, go to Runtime > Restart session before continuing.")

In [ ]:
# Clone the FinReasoningAI repository
# Replace the URL below with your actual GitHub repo URL after pushing
REPO_URL = "https://github.com/juankim834/FinReasoningAI.git"  # <-- update this

import os
PROJECT_DIR = "/content/FinReasoningAI"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    print(f"Repo already exists at {PROJECT_DIR}. Pulling latest...")
    !cd {PROJECT_DIR} && git pull

os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")
!ls

In [ ]:
# Create output directories (linked to Drive for persistence)
import os

dirs = [
    'data/raw',
    'data/processed',
    'outputs/sft_qlora',
    'outputs/dpo_qlora',
    'outputs/rag_index',
]
for d in dirs:
    drive_path = os.path.join(DRIVE_BASE, d)
    local_path = os.path.join(PROJECT_DIR, d)
    os.makedirs(drive_path, exist_ok=True)
    # Symlink local paths to Drive for persistence
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    if not os.path.exists(local_path):
        os.symlink(drive_path, local_path)

print("Directories created and linked to Drive.")

In [ ]:
# Log in to Hugging Face to access Qwen2.5-14B
# Get your token from: https://huggingface.co/settings/tokens
from huggingface_hub import login
from google.colab import userdata

# Store your token as a Colab secret named HF_TOKEN
# Colab > Secrets (key icon in left sidebar) > Add HF_TOKEN
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("Logged in to Hugging Face.")
except Exception:
    print("HF_TOKEN secret not found. Running login interactively...")
    login()  # prompts for token

---
## Step 1 — Load Model and Apply QLoRA

Loads Qwen2.5-14B-Instruct in 4-bit NF4 (double quantization) and attaches
LoRA adapters to all 7 projection modules.

**Memory after this cell:** approximately 9 GB (base weights) + 0.6 GB (LoRA) = ~10 GB

In [ ]:
import importlib.metadata
import torch
from src.model.load_model import load_model_and_tokenizer, DEFAULT_BNB_CONFIG

MODEL_ID = "Qwen/Qwen2.5-14B-Instruct"

# --- Sanity check bitsandbytes before loading ---
bnb_version = importlib.metadata.version("bitsandbytes")
print(f"bitsandbytes: {bnb_version}")
_major, _minor, *_ = bnb_version.split(".")
if int(_major) == 0 and int(_minor) < 44:
    raise RuntimeError(
        f"bitsandbytes {bnb_version} is too old (need >= 0.44.0).\n"
        "Fix: pip install -U 'bitsandbytes>=0.44.0'  then Runtime > Restart session."
    )

# --- Detect whether flash-attn is usable ---
try:
    import flash_attn
    ATTN_IMPL = "flash_attention_2"
    print(f"flash-attn {flash_attn.__version__} found. Using Flash Attention 2.")
except ImportError:
    ATTN_IMPL = "eager"
    print("flash-attn not installed. Using eager attention (slower but correct).")

# Load in 4-bit NF4 with double quantization.
# compute_dtype=bfloat16 gives wider dynamic range than fp16 for financial numbers.
model, tokenizer = load_model_and_tokenizer(
    model_id=MODEL_ID,
    bnb_config=DEFAULT_BNB_CONFIG,
    attn_implementation=ATTN_IMPL,
)

print(f"\nModel type  : {type(model).__name__}")
print(f"VRAM in use : {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
from src.model.apply_lora import apply_qlora, DEFAULT_LORA_CONFIG

# Attach QLoRA adapters
# Config: r=64, alpha=128, dropout=0.05
# Target modules: q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj
peft_model = apply_qlora(model, lora_config=DEFAULT_LORA_CONFIG, gradient_checkpointing=True)

print(f"VRAM after QLoRA: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
print(f"Model ready for training.")

---
## Step 2 — Data Generation and Preprocessing

Generates 5,000 synthetic FinQA samples, then formats and tokenizes them
into a HuggingFace DatasetDict (90% train / 5% val / 5% test).

**Data mix:**
- 60% Type A: Financial QA (context + question + answer)
- 30% Type B: Numerical Reasoning (expression + variables + result)
- 10% Type C: Structured Analysis (financial data table + CoT reasoning)

**Two generation modes:**
- `--no_llm`: Template-only, no GPU required, runs in ~30 seconds
- Default: Uses Qwen2.5-7B as generator (~16 GB VRAM, ~20 minutes)

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')

from src.data.synthetic_gen import generate_all_synthetic_data

# Set use_llm=True to use Qwen2.5-7B for higher quality Type A samples.
# Set use_llm=False for fast template-only generation (recommended for first run).
USE_LLM_GENERATOR = False  # <-- change to True for full quality

samples = generate_all_synthetic_data(
    output_path='data/raw/synthetic.jsonl',
    total_target=5000,
    numerical_fraction=0.30,
    use_llm=USE_LLM_GENERATOR,
    seed=42,
)

# Show sample breakdown
from collections import Counter
task_counts = Counter(s['task'] for s in samples)
print(f"\nGenerated {len(samples)} samples:")
for task, count in task_counts.items():
    print(f"  {task}: {count} ({count/len(samples)*100:.0f}%)")

In [ ]:
# Inspect a few samples from each type
import json

by_task = {}
for s in samples:
    by_task.setdefault(s['task'], []).append(s)

for task, task_samples in by_task.items():
    sample = task_samples[0]
    print(f"\n--- {task.upper()} ---")
    print(f"Question : {sample.get('question', '')[:120]}")
    print(f"Answer   : {sample.get('answer', '')[:80]}")
    if sample.get('expression'):
        print(f"Expression: {sample['expression']}")

In [ ]:
from src.data.preprocess import load_and_format_dataset, print_dataset_stats

# Format into ChatML and tokenize
# ChatML format used by Qwen2.5:
#   <|im_start|>system\n{system}<|im_end|>
#   <|im_start|>user\n{question}<|im_end|>
#   <|im_start|>assistant\n{answer}<|im_end|>

dataset = load_and_format_dataset(
    data_path='data/raw/synthetic.jsonl',
    tokenizer=tokenizer,
    max_length=2048,
    train_frac=0.90,
    val_frac=0.05,
    seed=42,
)

print_dataset_stats(dataset, tokenizer)
print(f"\nDataset splits: { {k: len(v) for k, v in dataset.items()} }")

# Save tokenized dataset to Drive for reuse
dataset.save_to_disk('data/processed')
print("Tokenized dataset saved to data/processed")

In [ ]:
# Show what a formatted ChatML prompt looks like
from src.data.preprocess import format_sample

example = samples[0]
formatted = format_sample(example)
print("=" * 60)
print("FORMATTED CHATML PROMPT EXAMPLE")
print("=" * 60)
print(formatted[:1000])
print("...")

---
## Step 3 — SFT Training

Trains the QLoRA model using TRL's SFTTrainer with completion-only loss masking
(loss computed only on the assistant response tokens, not the prompt).

**Training config:**
- Effective batch size: 4 x 8 = 32
- Learning rate: 2e-4 with cosine schedule + 50-step warmup
- Epochs: 3
- Optimizer: paged_adamw_32bit (offloads optimizer state to CPU)
- Precision: bf16

**Expected VRAM peak:** approximately 38-42 GB (well within A100 80 GB)

**Expected training time:** approximately 2-4 hours for 5,000 samples x 3 epochs on A100

In [ ]:
# Run full SFT training
# The main() function in sft_train.py wires everything together:
#   1. Load model + tokenizer
#   2. Apply QLoRA
#   3. Load dataset
#   4. Configure SFTTrainer with EarlyStoppingCallback
#   5. Train and save adapter

# NOTE: If you already ran apply_qlora() above, you can pass the peft_model
# directly to SFTTrainer instead of re-loading. For clarity, main() handles
# everything from scratch.

# Free the model loaded in Step 1 before re-loading in main()
import gc
del peft_model, model
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM freed. Current: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
from src.train.sft_train import main as sft_main

# CONFIGURATION — adjust to your needs
sft_main(
    model_id=MODEL_ID,
    output_dir='outputs/sft_qlora',
    data_dir='data/processed',
    num_train_epochs=3,
    per_device_train_batch_size=4,    # reduce to 2 if OOM on non-A100 GPUs
    gradient_accumulation_steps=8,    # effective batch = 4 x 8 = 32
    learning_rate=2e-4,
    max_seq_length=2048,
    use_wandb=False,                  # set True + add WANDB_API_KEY secret to enable W&B
)

print("\nSFT training complete. Adapter saved to outputs/sft_qlora/final_adapter")

### Step 3b — Optional: DPO Second Stage

DPO (Direct Preference Optimization) teaches the model to prefer grounded,
accurate answers over hallucinated ones. Run this after SFT if you observe
the model producing confident but wrong numbers.

**What it does:** Samples N completions from the SFT model, scores them
against ground truth, and trains on (chosen=best, rejected=worst) pairs.

**Skip this section** if SFT results are already satisfactory.

In [ ]:
# Build preference pairs from the SFT model
# This samples 8 completions per question and picks best/worst

SKIP_DPO = True  # <-- set to False to run DPO

if not SKIP_DPO:
    from datasets import load_from_disk
    from src.model.load_model import load_model_and_tokenizer, DEFAULT_BNB_CONFIG
    from peft import PeftModel
    from src.train.dpo_train import construct_preference_pairs_from_sft

    # Load SFT model
    sft_base, sft_tokenizer = load_model_and_tokenizer(MODEL_ID, DEFAULT_BNB_CONFIG)
    sft_model = PeftModel.from_pretrained(sft_base, 'outputs/sft_qlora/final_adapter')

    # Use the test split as evaluation samples
    ds = load_from_disk('data/processed')
    eval_samples = ds['test'].to_list()[:500]  # use up to 500 eval samples

    pairs = construct_preference_pairs_from_sft(
        sft_model=sft_model,
        tokenizer=sft_tokenizer,
        eval_samples=eval_samples,
        n_samples=8,
        output_path='data/raw/dpo_preferences.jsonl',
    )
    print(f"Generated {len(pairs)} preference pairs.")
else:
    print("DPO skipped (SKIP_DPO=True).")

In [ ]:
if not SKIP_DPO:
    from src.train.dpo_train import main as dpo_main

    dpo_main(
        model_id=MODEL_ID,
        sft_adapter_dir='outputs/sft_qlora/final_adapter',
        output_dir='outputs/dpo_qlora',
        pref_data_path='data/raw/dpo_preferences.jsonl',
        beta=0.1,          # stay close to SFT reference
        max_length=1024,
        num_train_epochs=1,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        learning_rate=5e-5,
    )
    print("DPO training complete.")

---
## Step 4 — Evaluation

Evaluates the fine-tuned model on the held-out test set.

**Metrics computed:**
- **Exact Match (EM):** Numeric answers matched within +/- 0.01% relative error
- **F1:** Token overlap for text answers (SQuAD-style)
- **Parsability rate:** Fraction of answers that contain an extractable number or entity
- **Grounding rate:** Fraction of numeric values in answers that are traceable to the context

Results are saved to a CSV for further analysis.

In [ ]:
import gc, torch, importlib.metadata
gc.collect()
torch.cuda.empty_cache()

from src.model.load_model import load_model_and_tokenizer, DEFAULT_BNB_CONFIG
from peft import PeftModel
import os

# Determine which adapter to evaluate
ADAPTER_DIR = (
    'outputs/dpo_qlora/final_adapter'
    if (not SKIP_DPO and os.path.exists('outputs/dpo_qlora/final_adapter'))
    else 'outputs/sft_qlora/final_adapter'
)
print(f"Evaluating adapter: {ADAPTER_DIR}")

# Reuse ATTN_IMPL detected in Step 1; fall back to eager if variable not defined
try:
    _attn = ATTN_IMPL
except NameError:
    try:
        import flash_attn
        _attn = "flash_attention_2"
    except ImportError:
        _attn = "eager"

eval_base, eval_tokenizer = load_model_and_tokenizer(
    MODEL_ID, DEFAULT_BNB_CONFIG, attn_implementation=_attn
)
eval_model = PeftModel.from_pretrained(eval_base, ADAPTER_DIR)
eval_model.eval()
print(f"Model loaded for evaluation. VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
from datasets import load_from_disk
from src.eval.evaluate import evaluate_model

ds = load_from_disk('data/processed')
test_ds = ds['test']
print(f"Test set size: {len(test_ds)} samples")

# Evaluate on up to 200 samples for speed; remove max_samples for full evaluation
metrics = evaluate_model(
    model=eval_model,
    tokenizer=eval_tokenizer,
    test_dataset=test_ds,
    output_csv='outputs/eval_results.csv',
    max_new_tokens=128,
    max_samples=200,
)

print("\nEvaluation Results:")
print("-" * 40)
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"  {k:<28}: {v:.4f}")
    else:
        print(f"  {k:<28}: {v}")

In [ ]:
# Inspect per-sample results
import pandas as pd

results_df = pd.read_csv('outputs/eval_results.csv')
print(f"Total evaluated: {len(results_df)} samples")
print(f"\nPer-task breakdown:")
print(results_df.groupby('task')[['exact_match', 'f1', 'parsable', 'grounding_rate']].mean().round(4))
print("\nLow-scoring samples (EM=0, for error analysis):")
display(results_df[results_df['exact_match'] == 0][
    ['task', 'question', 'ground_truth', 'prediction', 'f1', 'grounding_rate']
].head(10))

In [ ]:
# Run CPU-only robustness tests (metric correctness, no model needed)
!python -m pytest tests/test_robustness.py -k "not model_and_tokenizer" -v --tb=short -q

---
## Step 5 — Inference

Three inference modes are available:

| Mode | Description | When to use |
|------|-------------|-------------|
| Direct (default) | Greedy decoding, no CoT in output | Low-latency, simple questions |
| CoT | Full scratchpad in output | Debugging, complex multi-step chains |
| Self-consistency | N=5-10 samples, median/majority vote | Highest accuracy, batch/async workloads |
| Tool-augmented | Calculator + table parser + RAG | Arithmetic-heavy or document-lookup questions |

**Hallucination protection (always active):**
1. Numeric values in the answer are checked against the context (+/- 2%)
2. If a number is ungrounded, returns `"Insufficient information."`
3. Self-consistency confidence threshold: if < 30%, returns refusal

In [ ]:
from src.inference.generate import generate_answer, safe_calculate

# Sample financial context
CONTEXT = """
Apple Inc. reported total revenues of $394.3 billion for fiscal year 2022,
compared to $365.8 billion in 2021, representing growth of 7.8%.
Net income was $99.8 billion, with earnings per share of $6.11.
The company's operating margin was 30.3% and free cash flow totaled $111.4 billion.
Research and development expenses were $26.3 billion, or 6.7% of net sales.
""".strip()

In [ ]:
# Mode 1: Direct answer (default, greedy decoding)
question = "What was Apple's revenue growth rate from 2021 to 2022?"

answer = generate_answer(
    model=eval_model,
    tokenizer=eval_tokenizer,
    question=question,
    context=CONTEXT,
    use_cot=False,
    self_consistency_n=1,
    temperature=0.0,       # greedy
    max_new_tokens=128,
    grounding_check=True,
)

print(f"Question : {question}")
print(f"Answer   : {answer}")

In [ ]:
# Mode 2: Chain-of-thought (exposes the model's reasoning scratchpad)
question_cot = "What percentage of Apple's 2022 revenue was spent on R&D?"

answer_cot = generate_answer(
    model=eval_model,
    tokenizer=eval_tokenizer,
    question=question_cot,
    context=CONTEXT,
    use_cot=True,          # enables <think>...</think> output
    max_new_tokens=300,
)

print(f"Question : {question_cot}")
print(f"Answer   : {answer_cot}")

In [ ]:
# Mode 3: Self-consistency (8 samples, median for numbers, majority vote for text)
# More accurate than greedy for complex numerical questions; ~8x slower
question_sc = "What was Apple's free cash flow in fiscal year 2022?"

answer_sc = generate_answer(
    model=eval_model,
    tokenizer=eval_tokenizer,
    question=question_sc,
    context=CONTEXT,
    self_consistency_n=8,  # sample 8 times
    temperature=0.7,       # must be > 0 for diversity
    max_new_tokens=128,
    min_confidence=0.30,   # return refusal if < 30% agreement
)

print(f"Question : {question_sc}")
print(f"Answer   : {answer_sc}")

In [ ]:
# Mode 4: Tool-augmented (model can call calculate() during generation)
# The model emits <tool_call>calculate(expr)</tool_call> tags,
# which are intercepted and evaluated by the Python interpreter.

question_tool = "Calculate Apple's R&D as a percentage of net income in 2022."

answer_tool = generate_answer(
    model=eval_model,
    tokenizer=eval_tokenizer,
    question=question_tool,
    context=CONTEXT,
    use_tools=True,       # enables tool-call interception
    max_new_tokens=256,
)

print(f"Question : {question_tool}")
print(f"Answer   : {answer_tool}")

# Standalone calculator (no model needed)
result = safe_calculate("26.3 / 99.8 * 100")
print(f"\nCalculator verification: 26.3 / 99.8 * 100 = {result:.2f}%")

---
## Step 6 — Self-Consistency Internals

This section demonstrates the self-consistency aggregation logic directly,
without requiring a model. Useful for understanding and testing the aggregation behavior.

In [ ]:
from src.inference.self_consistency import self_consistent_answer

# Numerical aggregation: uses median, robust to outliers
numerical_samples = [
    "$111.4 billion",
    "111.4B",
    "$111.5 billion",
    "approximately $111.4 billion",
    "$111.4B",
    "$111.3 billion",
    "$200 billion",     # outlier hallucination
    "$111.4 billion",
]

final_num, confidence_num = self_consistent_answer(numerical_samples)
print(f"Numerical aggregation:")
print(f"  Samples   : {numerical_samples}")
print(f"  Final     : {final_num}")
print(f"  Confidence: {confidence_num:.0%}")
print(f"  (Note: $200B outlier is ignored by median aggregation)")

print()

# Categorical aggregation: majority vote
text_samples = [
    "Apple's operating margin improved",
    "The operating margin improved year over year",
    "Operating margins declined",
    "Apple's operating margin improved",
    "The margin improved significantly",
]

final_cat, confidence_cat = self_consistent_answer(text_samples)
print(f"Categorical aggregation:")
print(f"  Samples   : {text_samples}")
print(f"  Final     : {final_cat}")
print(f"  Confidence: {confidence_cat:.0%}")

---
## Step 7 — Extensions: RAG and ReAct Agent

### 7a: RAG (Retrieval-Augmented Generation)

Builds a FAISS vector index over your financial documents using bge-m3 embeddings.
At query time: retrieves top-20 chunks, reranks to top-2 with a cross-encoder,
then passes the assembled context to the model.

**When to use RAG:**
- Questions that require information from specific 10-K filings or earnings transcripts
- The model does not have training-time knowledge of the relevant company/period
- You want citations / source traceability for the answer

**Install additional RAG dependencies:**

In [ ]:
!pip install -q faiss-cpu FlagEmbedding sentence-transformers

In [ ]:
from src.rag.retriever import BGEEmbedder, FinancialRetriever, chunk_document

# Create a small demo corpus from our sample context
DEMO_DOCS = [
    {
        "text": CONTEXT,
        "metadata": {"company": "Apple", "year": "2022",
                     "filing_type": "10-K", "section": "Financial Highlights"}
    },
    {
        "text": (
            "Microsoft Corporation reported total revenue of $198.3 billion for fiscal year 2022, "
            "up 18% from $168.1 billion in fiscal 2021. Operating income grew 20% to $83.4 billion. "
            "The cloud segment Azure grew 40% year over year. Net income was $72.7 billion."
        ),
        "metadata": {"company": "Microsoft", "year": "2022",
                     "filing_type": "10-K", "section": "Financial Highlights"}
    },
]

# Chunk documents
all_chunks = []
for doc in DEMO_DOCS:
    chunks = chunk_document(doc['text'], doc['metadata'], chunk_size=128, chunk_overlap=16)
    all_chunks.extend(chunks)

print(f"Created {len(all_chunks)} chunks from {len(DEMO_DOCS)} documents")

# Build FAISS index
embedder = BGEEmbedder(model_name="BAAI/bge-m3")
retriever = FinancialRetriever(embedder=embedder)
retriever.build_index(all_chunks, index_type="flat")

# Save to Drive
retriever.save('outputs/rag_index')
print("RAG index saved.")

In [ ]:
from src.rag.retriever import rag_answer

# Query the index
rag_question = "What was Apple's free cash flow in 2022?"

# Retrieve without model (just show retrieved context)
chunks = retriever.retrieve_and_rerank(rag_question, top_k=5, top_r=2)
context = retriever.format_context(chunks)
print(f"Query: {rag_question}")
print(f"\nRetrieved context:")
print("-" * 50)
print(context)

# Full RAG answer (requires eval_model from Step 4)
# answer, chunks = rag_answer(retriever, eval_model, eval_tokenizer, rag_question)
# print(f"\nRAG Answer: {answer}")

### 7b: Tool Router — Calculator and Table Parser

The ToolRouter provides a safe Python arithmetic evaluator and a
markdown/CSV table parser. These can be called directly or via the
ReAct agent loop.

In [ ]:
from src.tools.tool_router import ToolRouter
import json

router = ToolRouter(retriever=retriever)  # pass retriever to enable RAG tool

# Tool 1: Calculator
expr = "(394.3 - 365.8) / 365.8 * 100"
result = router.execute("calculate", expr)
print(f"calculate({expr}) = {result}")

# Tool 2: Table parser
md_table = """
| Year | Revenue ($B) | Net Income ($B) | EPS   | Op Margin |
|------|-------------|-----------------|-------|-----------|
| 2022 | 394.3       | 99.8            | 6.11  | 30.3%     |
| 2021 | 365.8       | 94.7            | 5.61  | 29.8%     |
| 2020 | 274.5       | 57.4            | 3.28  | 24.1%     |
""".strip()

table_result = router.execute("parse_table", f"{repr(md_table)}, 'markdown'")
parsed = json.loads(table_result)
print(f"\nparse_table result:")
print(f"  Summary: {parsed['summary']}")
print(f"  Columns: {parsed['columns']}")
print(f"  Row 0  : {parsed['rows'][0]}")

# Tool 3: Retrieve
retrieved = router.execute("retrieve", "Apple 2022 free cash flow")
print(f"\nretrieve result:\n{retrieved[:300]}")

### 7c: ReAct Agent Loop

The ReAct agent follows: Thought -> Action -> Observation -> (repeat) -> Answer

- Maximum 5 reasoning steps to bound latency
- Uses the model to generate each Thought + Action
- Tool outputs (Observations) are injected back into context
- If the model outputs `Answer:`, the loop terminates immediately

In [ ]:
from src.tools.tool_router import react_agent

# The agent will:
#   Step 1: Thought: I need to calculate 26.3 / 99.8 * 100
#   Step 1: Action: calculate(26.3 / 99.8 * 100)
#   Step 1: Observation: 26.35...
#   Step 2: Answer: Apple spent 26.35% of net income on R&D.

agent_question = "What percentage of Apple's net income was spent on R&D in 2022?"

final_answer, trace = react_agent(
    model=eval_model,
    tokenizer=eval_tokenizer,
    question=agent_question,
    context=CONTEXT,
    tool_router=router,
    max_steps=5,
    temperature=0.2,
)

print(f"Question: {agent_question}")
print(f"\nReasoning trace:")
for step in trace:
    print(f"  Step {step['step']} [{step['type']}]:")
    if step['type'] == 'action':
        print(f"    Tool: {step['tool']}({step['args'][:60]})")
        print(f"    Observation: {step['observation'][:80]}")
    else:
        print(f"    {str(step.get('content', step.get('thought', '')))[:120]}")

print(f"\nFinal answer: {final_answer}")

---
## Step 8 — Save and Export

Export the trained adapter for deployment or further use.

In [ ]:
# Option A: Keep as LoRA adapter (smallest, fastest to load)
# The adapter is already saved at outputs/sft_qlora/final_adapter
# and symlinked to Drive. Nothing extra to do.

import os
adapter_files = os.listdir('outputs/sft_qlora/final_adapter')
print("Adapter files saved:")
for f in adapter_files:
    size_mb = os.path.getsize(f'outputs/sft_qlora/final_adapter/{f}') / 1e6
    print(f"  {f}: {size_mb:.1f} MB")

In [ ]:
# Option B: Merge LoRA into base weights (required for vLLM serving)
# NOTE: Merged model is approximately 28GB (bfloat16) -- ensure enough Drive space

MERGE_FOR_SERVING = False  # <-- set to True to merge

if MERGE_FOR_SERVING:
    from src.model.load_model import load_model_and_tokenizer, DEFAULT_BNB_CONFIG
    from peft import PeftModel
    try:
        _attn_merge = ATTN_IMPL
    except NameError:
        _attn_merge = "eager"

    print("Loading base model for merging...")
    base, tok = load_model_and_tokenizer(
        MODEL_ID, DEFAULT_BNB_CONFIG, attn_implementation=_attn_merge
    )
    peft_m = PeftModel.from_pretrained(base, 'outputs/sft_qlora/final_adapter')

    print("Merging LoRA weights into base model...")
    merged = peft_m.merge_and_unload()

    MERGED_DIR = 'outputs/merged_model'
    merged.save_pretrained(MERGED_DIR)
    tok.save_pretrained(MERGED_DIR)
    print(f"Merged model saved to {MERGED_DIR}")
    print("Load with vLLM: vllm serve outputs/merged_model")
else:
    print("Merge skipped (MERGE_FOR_SERVING=False).")

In [ ]:
# Option C: Push adapter to Hugging Face Hub (private repo)
PUSH_TO_HUB = False  # <-- set to True to push
HF_REPO_NAME = "your-username/finreasoningai-qlora"  # <-- update this

if PUSH_TO_HUB:
    eval_model.push_to_hub(HF_REPO_NAME, private=True)
    eval_tokenizer.push_to_hub(HF_REPO_NAME, private=True)
    print(f"Adapter pushed to: https://huggingface.co/{HF_REPO_NAME}")
else:
    print("Hub push skipped (PUSH_TO_HUB=False).")

---
## Quick Reference

### Key configuration parameters

| Parameter | Default | Where to change |
|-----------|---------|----------------|
| Base model | Qwen/Qwen2.5-14B-Instruct | `MODEL_ID` in Step 1 |
| LoRA rank | 64 | `DEFAULT_LORA_CONFIG.r` in apply_lora.py |
| Batch size | 4 (x8 grad accum = 32 effective) | `per_device_train_batch_size` in sft_main() |
| Learning rate | 2e-4 | `learning_rate` in sft_main() |
| Max sequence length | 2048 tokens | `max_seq_length` in sft_main() |
| Self-consistency N | 1 (greedy) | `self_consistency_n` in generate_answer() |
| DPO beta | 0.1 | `beta` in dpo_main() |

### Troubleshooting

| Problem | Fix |
|---------|-----|
| CUDA out of memory during training | Reduce `per_device_train_batch_size` to 2, or `max_seq_length` to 1024 |
| Flash Attention 2 install fails | Pass `attn_implementation="eager"` to `load_model_and_tokenizer()` |
| Model returns "Insufficient information" too often | Set `grounding_check=False` in `generate_answer()` |
| Low self-consistency confidence | Increase `self_consistency_n` or lower `min_confidence` |
| Training loss not decreasing | Check that `data/processed` was generated from the correct tokenizer |

### Inference modes at a glance

```python
from src.inference.generate import generate_answer

# Greedy (fastest, deterministic)
answer = generate_answer(model, tokenizer, question, context)

# Chain-of-thought (shows reasoning)
answer = generate_answer(model, tokenizer, question, context, use_cot=True)

# Self-consistency (most accurate)
answer = generate_answer(model, tokenizer, question, context,
                         self_consistency_n=8, temperature=0.7)

# Tool-augmented (calculator + table parser)
answer = generate_answer(model, tokenizer, question, context, use_tools=True)
```